In [9]:
import open3d as o3d
import numpy as np

import numpy as np

def compute_voxel_ece(labels_1_matched, labels_2_matched, uncertainties_matched, matched_indices, voxel_res=0.04, num_bins=10):
  """
  Computes voxel-wise Expected Calibration Error (ECE).
  
  Args:
      pcd_1 (str): Path to first point cloud (ground truth).
      labels_1 (np.ndarray): Labels for pcd_1.
      pcd_2 (str): Path to second point cloud (predicted).
      labels_2 (np.ndarray): Labels for pcd_2.
      uncertainties (np.ndarray): Uncertainty values for pcd_2.
      voxel_res (float): Voxel resolution threshold.
      num_bins (int): Number of bins for ECE calculation.
  
  Returns:
      float: Expecten_classesd Calibration Error (ECE).
  """
  # Step 1: Match points and get corresponding labels & uncertainties

  # Ensure we have valid matches
  if len(matched_indices) == 0:
      return 0.0  # No matches → No calibration error

  # Step 2: Convert uncertainties to confidence (1 - uncertainty)
  confidences = 1 - uncertainties_matched

  # Step 3: Compute correctness
  correctness = (labels_1_matched == labels_2_matched).astype(float)  # 1 if correct, 0 if incorrect

  # Step 4: Create bins
  bin_edges = np.linspace(0, 1, num_bins + 1)
  bin_indices = np.digitize(confidences, bin_edges) - 1  # Assign each point to a bin

  ece = 0.0
  total_samples = len(matched_indices)

  # Step 5: Compute ECE
  for i in range(num_bins):
      bin_mask = bin_indices == i
      bin_count = np.sum(bin_mask)

      if bin_count > 0:
          avg_confidence = np.mean(confidences[bin_mask])
          avg_accuracy = np.mean(correctness[bin_mask])
          ece += (bin_count / total_samples) * abs(avg_confidence - avg_accuracy)

  return ece


def find_pcd_matches(pcd_1, labels_1, pcd_2, labels_2, uncertainties=None, voxel_res=0.04):
    pcd_2 = o3d.io.read_point_cloud(pcd_2)
    pcd_tree = o3d.geometry.KDTreeFlann(pcd_2)
    pcd_1 = o3d.io.read_point_cloud(pcd_1)
    
    num_1 = np.asarray(pcd_1.points).shape[0]
    to_2 = np.full(num_1, -1, dtype=int)

    points_1 = np.asarray(pcd_1.points)
    for i, point in enumerate(points_1):
        [k, idx, dists] = pcd_tree.search_knn_vector_3d(point, 1)
        if k > 0 and dists[0] < voxel_res:
            to_2[i] = idx[0]

    matched_indices = np.where(to_2 != -1)[0]
    labels_1_matched = labels_1[matched_indices]
    labels_2_matched = labels_2[to_2[matched_indices]]
    uncertainties_matched = uncertainties[to_2[matched_indices]] if uncertainties is not None else None


    return labels_1_matched, labels_2_matched, uncertainties_matched, matched_indices

def read_pcd(file_path):
    """
    Reads an ASCII PCD file and returns point coordinates, labels,
    and optionally an uncertainty field if present.
    
    This function assumes the PCD file has a header with a "FIELDS" line.
    The expected order is:
      - x, y, z, labebase_output_dirl[, uncertainty]
    
    Args:
        file_path (str): Path to the PCD file.
    
    Returns:
        tuple: (points, labels, uncertainty)
            - points: (N, 3) numpy array of float32.
            - labels: (N,) numpy array of int32.
            - uncertainty: (N,) numpy array of float32 if present, otherwise None.
    """
    with open(file_path, 'r') as f:
        lines = f.readlines()

    header_lines = []
    data_start = 0
    for i, line in enumerate(lines):
        line = line.strip()
        if not line:
            continue
        header_lines.append(line)
        if line.startswith("DATA"):
            data_start = i + 1
            break

    # Extract field names from header if available.
    fields = []
    for line in header_lines:
        if line.startswith("FIELDS"):
            parts = line.split()
            fields = parts[1:]
            break
    # Fallback: assume default fields if not found.
    if not fields:
        fields = ["x", "y", "z", "label"]

    # Load data lines.
    data = []
    for line in lines[data_start:]:
        if line.strip():
            data.append(line.strip().split())
    data = np.array(data)

    # Parse points (first 3 columns) and labels (4th column).
    points = data[:, :3].astype(np.float32)
    labels = data[:, 3].astype(np.int32) if data.shape[1] >= 4 else np.zeros(data.shape[0], dtype=np.int32)
    
    # Optionally parse uncertainty if available.
    uncertainty = None
    if data.shape[1] >= 5 or "uncertainty" in fields:
        uncertainty = data[:, 4].astype(np.float32)
    
    return points, labels, uncertainty


In [10]:
def chamfer_distance(pcd_1_path, pcd_2_path):
    pcd_1 = o3d.io.read_point_cloud(pcd_1_path)
    pcd_2 = o3d.io.read_point_cloud(pcd_2_path)
    # Convert Open3D point clouds to NumPy arrays
    points_1 = np.asarray(pcd_1.points)
    points_2 = np.asarray(pcd_2.points)

    # Build KD-Trees for fast nearest neighbor search
    pcd_tree_1 = o3d.geometry.KDTreeFlann(pcd_1)
    pcd_tree_2 = o3d.geometry.KDTreeFlann(pcd_2)

    # Compute Chamfer Distance (one direction)
    dist_1_to_2 = []
    for p in points_1:
        [_, idx, dists] = pcd_tree_2.search_knn_vector_3d(p, 1)
        dist_1_to_2.append(dists[0])

    dist_2_to_1 = []
    for p in points_2:
        [_, idx, dists] = pcd_tree_1.search_knn_vector_3d(p, 1)
        dist_2_to_1.append(dists[0])

    # Compute final Chamfer Distance
    chamfer_dist = np.mean(dist_2_to_1)
    return chamfer_dist





In [11]:
!pip install pot
import ot

def earth_mover_distance(points1, points2):
  """
  Compute the Earth Mover's Distance (EMD) between two point clouds.

  Args:
      points1 (np.ndarray): N x 3 array of points (point cloud 1).
      points2 (np.ndarray): M x 3 array of points (point cloud 2).

  Returns:
      float: The Earth Mover's Distance.
  """
  n, m = len(points1), len(points2)

  # Compute pairwise Euclidean distance matrix
  cost_matrix = np.linalg.norm(points1[:, None, :] - points2[None, :, :], axis=2)

  # Uniform weights (each point contributes equally)
  weight1 = np.ones(n) / n
  weight2 = np.ones(m) / m

  # Compute EMD using POT's emd2 function
  emd_value = ot.emd2(weight1, weight2, cost_matrix)

  return emd_value

In [12]:
import open3d as o3d
import numpy as np
from scipy.spatial.distance import directed_hausdorff

# def load_ply_points(file_path):
#   """Load a PLY file and return an Nx3 numpy array of points."""
#   pcd = o3d.io.read_point_cloud(file_path)
#   return np.asarray(pcd.points)

def hausdorff_distance(points1, points2):
  """Compute the Hausdorff distance between two point clouds."""


  # Compute directed Hausdorff distance in both directions
  d1 = directed_hausdorff(points1, points2)[0]
  d2 = directed_hausdorff(points2, points1)[0]

  # The Hausdorff distance is the maximum of the two
  return d1 # max(d1, d2)



In [13]:
import numpy as np
import trimesh
import scipy.sparse
from scipy.spatial import KDTree

def compute_laplacian_smoothness_error(mesh_gt, mesh_rec):
  """Computes the Laplacian smoothness error between GT and reconstructed mesh (supports PLY)."""
  
  def compute_laplacian(mesh):
    """Computes the Laplacian of the mesh using cotangent weights."""
    n = len(mesh.vertices)
    L = scipy.sparse.lil_matrix((n, n))

    for face in mesh.faces:
      for i in range(3):
        vi = face[i]
        vj = face[(i + 1) % 3]
        vk = face[(i + 2) % 3]

        e0 = mesh.vertices[vj] - mesh.vertices[vi]
        e1 = mesh.vertices[vk] - mesh.vertices[vi]
        e2 = mesh.vertices[vk] - mesh.vertices[vj]

        cot_alpha = np.dot(e0, e1) / np.linalg.norm(np.cross(e0, e1))
        cot_beta = np.dot(-e0, e2) / np.linalg.norm(np.cross(-e0, e2))

        w = 0.5 * (cot_alpha + cot_beta)

        L[vi, vj] -= w
        L[vj, vi] -= w
        L[vi, vi] += w
        L[vj, vj] += w

    return L @ mesh.vertices  # Laplacian coordinates

  # Compute Laplacian for both meshes
  lap_gt = compute_laplacian(mesh_gt)
  lap_rec = compute_laplacian(mesh_rec)

  # Map GT Laplacian values to reconstructed mesh using nearest neighbors
  tree = KDTree(mesh_gt.vertices)
  _, indices = tree.query(mesh_rec.vertices)

  lap_gt_mapped = lap_gt[indices]  # Interpolated values

  error = np.linalg.norm(lap_gt_mapped - lap_rec, axis=1).mean()
  return error

def compute_mean_curvature(mesh):
  """Computes mean curvature using the Laplacian of vertex positions."""
  L = mesh.laplacian
  curvature = np.linalg.norm(L @ mesh.vertices, axis=1)
  return curvature

def compute_mean_curvature_deviation(mesh_gt, mesh_rec):
  """Computes the mean curvature deviation between GT and reconstructed mesh (supports PLY)."""
  curv_gt = compute_mean_curvature(mesh_gt)
  curv_rec = compute_mean_curvature(mesh_rec)

  # Use nearest neighbors to map GT curvature to reconstructed mesh
  tree = KDTree(mesh_gt.vertices)
  _, indices = tree.query(mesh_rec.vertices)

  curv_gt_mapped = curv_gt[indices]  # Interpolated values

  error = np.abs(curv_gt_mapped - curv_rec).mean()
  return error



In [14]:
import numpy as np
import csv

import numpy as np
import csv

class SceneMetricTracker:
  def __init__(self, class_names):
    self.class_names = class_names  # List of class names
    self.n_classes = len(class_names)
    self.total_intersections = np.zeros(self.n_classes, dtype=np.float64)
    self.total_unions = np.zeros(self.n_classes, dtype=np.float64)
    self.total_chamfer_dist = 0.0
    self.total_correct_voxels = 0
    self.total_voxels = 0
    self.scene_metrics = []
    self.scene_count = 0
    self.total_uce = 0.0
    self.total_hausdorff_dist = 0.0
    self.total_ece = 0.0

  def update_scene(self, pred, gt, labels_gt, chamfer_dist, scene_name, uncertainties, pred_pcd, gt_pcd, ece):#, pred_mesh_path, gt_mesh_path): #, curvature_error)
    """
    Computes IoU, voxel accuracy, and chamfer distance for a scene.
    labels_gt contains all labels present in the full ground truth point cloud.
    """
    scene_intersections = np.zeros(self.n_classes, dtype=np.float64)
    scene_unions = np.zeros(self.n_classes, dtype=np.float64)
    scene_ious = np.full(self.n_classes, np.nan, dtype=np.float64)  # Initialize with NaN

    # Only compute IoU for classes that exist in labels_gt
    unique_classes_in_gt = np.unique(labels_gt)

    for i in unique_classes_in_gt:
      if i >= self.n_classes:  # Ignore unknown class IDs
        continue
      
      if i ==21 or i == 22 or i == 23: # Ignore class 0 (background) and any other specific classes
        continue

      gt_mask = gt == i
      pred_mask = pred == i

      intersection = np.logical_and(gt_mask, pred_mask).sum()
      union = np.logical_or(gt_mask, pred_mask).sum()

      scene_intersections[i] = intersection
      scene_unions[i] = union

      scene_ious[i] = intersection / (union + 1e-8) if union > 0 else 0.0

      # Update cumulative IoU
      self.total_intersections[i] += intersection
      self.total_unions[i] += union

    # Compute per-scene mIoU **ignoring NaNs**
    scene_miou = np.nanmean(scene_ious) if np.any(~np.isnan(scene_ious)) else np.nan

    # Voxel Accuracy
    correct_voxels = np.sum(pred == gt)
    total_voxels = gt.size
    scene_voxel_acc = correct_voxels / total_voxels if total_voxels > 0 else 0.0

    # # Ensure valid indices are within gt_pcd shape range
    # valid_mask = (indices >= 0) & (indices < gt_pcd.shape[0])  # Validate against gt_pcd, not gt
    # indices = indices[valid_mask]  # Keep only valid indices
    
    # matched_pred = pred[valid_mask]
    # matched_uncertainties = uncertainties[valid_mask]
    # matched_gt_labels = gt[indices] if len(indices) > 0 else np.array([])  # Ensure valid lookup

    # # Compute UCE only if matched_gt_labels is non-empty
    # uce = 0.0
    # if len(matched_gt_labels) > 0:
    #     num_bins = 10
    #     bins = np.linspace(0, 1, num_bins + 1)  # Define bin edges
    #     bin_indices = np.digitize(matched_uncertainties, bins) - 1  # Assign voxels to bins
        
    #     for i in range(num_bins):
    #         bin_mask = bin_indices == i
    #         bin_size = np.sum(bin_mask)
    #         if bin_size == 0:
    #             continue

    #         error_bin = 1.0 - np.mean(matched_pred[bin_mask] == matched_gt_labels[bin_mask])  # Error in bin
    #         unc_bin = np.mean(matched_uncertainties[bin_mask])  # Mean uncertainty in bin

    #         uce += (bin_size / len(matched_pred)) * np.abs(error_bin - unc_bin)  # Weighted difference

    #hausdorff_distance_metric = hausdorff_distance(pred_pcd, gt_pcd)
    #emd = earth_mover_distance(pred_pcd, gt_pcd)
    # Update cumulative values
    self.total_correct_voxels += correct_voxels
    self.total_voxels += total_voxels
    self.total_chamfer_dist += chamfer_dist
    self.scene_count += 1
    self.total_ece += ece

    # Store per-scene metrics
    self.scene_metrics.append({
      "scene_id": scene_name,
      "per_class_ious": scene_ious.tolist(),
      "scene_miou": scene_miou,
      "scene_chamfer_dist": chamfer_dist,
      "scene_voxel_acc": scene_voxel_acc,
      "scene_voxel_ece": ece      #"laplacian_error": laplacian_error,
      #"curvature_error": curvature_error
    })

  def get_overall_classwise_ious(self):
    """Returns cumulative IoU per class across all scenes (ignores NaN)."""
    overall_ious = np.full(self.n_classes, np.nan, dtype=np.float64)  # Initialize with NaN
    valid_classes = self.total_unions > 0  # Only compute for classes that exist
    overall_ious[valid_classes] = self.total_intersections[valid_classes] / (self.total_unions[valid_classes] + 1e-8)
    return overall_ious

  def get_final_mIoU(self):
    """Computes cumulative mIoU as the mean of cumulative per-class IoUs, ignoring NaN."""
    overall_ious = self.get_overall_classwise_ious()
    # Exclude class 0 (index 0) for final mIoU calculation
    overall_ious_excluding_class_0 = overall_ious[1:]
    return np.nanmean(overall_ious_excluding_class_0) if np.any(~np.isnan(overall_ious_excluding_class_0)) else np.nan

  def get_final_voxel_acc(self):
    """Computes overall voxel accuracy."""
    return self.total_correct_voxels / self.total_voxels if self.total_voxels > 0 else 0.0

  def get_final_chamfer_dist(self):
    """Computes overall mean Chamfer distance."""
    return self.total_chamfer_dist / self.scene_count if self.scene_count > 0 else 0.0
  
  def get_final_uce(self):
    """Computes overall UCE."""
    return self.total_uce / self.scene_count if self.scene_count > 0 else 0.0
  
  def get_final_hausdorff_dist(self):
    """Computes overall Hausdorff distance."""
    return self.total_hausdorff_dist / self.scene_count if self.scene_count > 0 else 0.0

  def get_final_emd(self):
    """Computes overall Earth Mover's Distance."""
    return self.total_emd / self.scene_count if self.scene_count > 0 else 0.0
  
  def get_final_ece(self):
    """Computes overall ECE."""
    return self.total_ece / self.scene_count if self.scene_count > 0 else 0.0

  def write_csv(self, filename):
    """Writes per-scene and cumulative metrics to a CSV file using class names and NaNs for missing classes."""
    with open(filename, mode="w", newline="") as file:
      writer = csv.writer(file)

      # CSV Header
      header = ["Scene ID"] + [f"IoU_{class_name}" for class_name in self.class_names] + [
        "Scene_mIoU", "Scene_Chamfer_Dist", "Scene_Voxel_Acc", "Scene_Voxel_ECE", "hausdorff_dist", "emd"
      ]
      writer.writerow(header)

      # Per-Scene Metrics
      for scene in self.scene_metrics:
        row = [scene["scene_id"]] + scene["per_class_ious"] + [
          scene["scene_miou"], scene["scene_chamfer_dist"], scene["scene_voxel_acc"], scene["scene_voxel_ece"]
        ]
        writer.writerow(row)

      # Cumulative Metrics Row
      overall_ious = self.get_overall_classwise_ious().tolist()
      final_miou = self.get_final_mIoU()
      final_chamfer_dist = self.get_final_chamfer_dist()
      final_voxel_acc = self.get_final_voxel_acc()
      final_ece = self.get_final_ece()


      writer.writerow(["Overall"] + overall_ious + [final_miou, final_chamfer_dist, final_voxel_acc, final_ece])

      #writer.writerow(["Overall"] + overall_ious + [final_miou, final_chamfer_dist, final_voxel_acc])

    print(f"Metrics saved to {filename}")



In [15]:
import os
ROOT_DIR = "/home/user/menon/workspace/ros1/menon25iros_ws/src/kimera_semantics_evidential/kimera_semantics_evidential_ros/evaluation_results"
SCANNET_GROUNDTRUTH_FOLDER = "scannet22_scans"
EXP_TYPE = "scannet22_swinmtl_onehot_evid"
SCENES = ["scene0013_00", "scene0019_00", "scene0115_02", "scene0369_02", "scene0480_01"]
#class_names = ["BG", "Bed", "Books", "Ceiling", "Chair", "Floor", "Furniture",
#               "Objects", "Picture", "Sofa", "Table", "TV", "Wall", "Window"]
# Initialize a SceneMetricTracker with the number of classes
n_classes = 23
class_names = [
    "background",
    "wall",
    "floor",
    "cabinet",
    "bed",
    "chair",
    "sofa",
    "table",
    "door",
    "window",
    "bookshelf",
    "picture",
    "counter",
    "desk",
    "curtain",
    "refridgerator",
    "showercurtain",
    "toilet",
    "sink",
    "bathtub",
    "otherfurniture",
    "otherstruct",
    "otherprop",
    "freespace",
]
tracker = SceneMetricTracker(class_names)

for scene in SCENES:
    print(f"Processing scene {scene}")
    gt_pcd_path = f"{ROOT_DIR}/{SCANNET_GROUNDTRUTH_FOLDER}/{SCANNET_GROUNDTRUTH_FOLDER}_{scene}.pcd"
    pred_pcd_path = f"{ROOT_DIR}/{EXP_TYPE}/{EXP_TYPE}_{scene}.pcd"
    gt_points, gt_labels, _ = read_pcd(gt_pcd_path)
    if os.path.exists(pred_pcd_path):
        pred_points, pred_labels, uncertainties = read_pcd(pred_pcd_path)
    else:
        print(f"Skipping scene {scene} since prediction file not found")
        continue
    print("Read PCD files for scene successfully", scene)
    chamfer_dist = chamfer_distance(gt_pcd_path, pred_pcd_path)
    print("Computed Chamfer distance successfully", scene)
    pred_labels_matched, gt_labels_matched, uncertainties_matched, matched_indices = find_pcd_matches(pred_pcd_path, pred_labels, gt_pcd_path, gt_labels, uncertainties)
    print("Computed PCD matches successfully", scene)
    gt_mesh_path = f"{ROOT_DIR}/{SCANNET_GROUNDTRUTH_FOLDER}/{SCANNET_GROUNDTRUTH_FOLDER}_{scene}.ply"
    pred_mesh_path = f"{ROOT_DIR}/{EXP_TYPE}/{EXP_TYPE}_{scene}.ply"
    voxel_ece = compute_voxel_ece(gt_labels_matched, pred_labels_matched, uncertainties_matched, matched_indices)
    #mesh_gt = trimesh.load(gt_mesh_path)
    #mesh_pred = trimesh.load(pred_mesh_path)

    # Compute smoothness errors
    #laplacian_error = compute_laplacian_smoothness_error(mesh_gt, mesh_pred)
    #curvature_error = compute_mean_curvature_deviation(mesh_gt, mesh_pred)

    tracker.update_scene(pred_labels_matched, gt_labels_matched, gt_labels, chamfer_dist, scene, uncertainties, pred_points, gt_points, voxel_ece)#, pred_pcd_path, pred_pcd_path) #, curvature_error)
    print("Updated scene metrics successfully", scene)

RESULTS_FILE = f"{ROOT_DIR}/{EXP_TYPE}_results.csv"
tracker.write_csv(RESULTS_FILE)
print(f"Results saved to {RESULTS_FILE}")


Processing scene scene0013_00
Read PCD files for scene successfully scene0013_00
Computed Chamfer distance successfully scene0013_00
Computed PCD matches successfully scene0013_00
Updated scene metrics successfully scene0013_00
Processing scene scene0019_00
Read PCD files for scene successfully scene0019_00
Computed Chamfer distance successfully scene0019_00
Computed PCD matches successfully scene0019_00
Updated scene metrics successfully scene0019_00
Processing scene scene0115_02
Read PCD files for scene successfully scene0115_02
Computed Chamfer distance successfully scene0115_02
Computed PCD matches successfully scene0115_02
Updated scene metrics successfully scene0115_02
Processing scene scene0369_02
Read PCD files for scene successfully scene0369_02
Computed Chamfer distance successfully scene0369_02
Computed PCD matches successfully scene0369_02
Updated scene metrics successfully scene0369_02
Processing scene scene0480_01
Read PCD files for scene successfully scene0480_01
Compute